# A2 · The components with the power method: convergence and the gap between eigenvalues

**Type**: Implement from scratch · **Level**: ●●● challenge · **Time**: about 55 minutes · **extended path**

## Objective

Compute the first principal components with the **power method** (repeatedly multiplying a vector by the covariance matrix) and **deflation** (removing the component found), check the equality with `np.linalg.eigh`, and measure that the speed of
convergence depends on the ratio $\lambda_2/\lambda_1$ (the spectral gap).

## Dataset

`wine_C` (13 × 13): correlation matrix of the wines. `narrow_C` (13 × 13): a symmetric matrix built with the same spectrum except the first two eigenvalues, made almost equal (ratio 0.98).

## Educational purpose

Understand how the components are computed when the matrix is huge (the power method is at the base of the randomized algorithms and of those for sparse matrices), and understand what the speed depends on: if two components have almost equal variances, telling them apart is hard
(and numerically unstable).

## Task

1. `power(M, tol, max_iter, seed)` → `(v, lam, n_iter, errors)`: start from a normalized random vector (`np.random.default_rng(seed).normal`), repeat `v ← M v / ‖M v‖` until the angle with the previous iteration is below `tol` (`1 − |v_new · v_old| < tol`), and return the eigenvector `v`, the eigenvalue `lam = v @ M @ v`, the number of iterations and the list of errors `errors[t] = 1 − |v_t · true_v|` (where `true_v` is the true component, passed as an optional argument `true_v=None`; if it is `None`, don't record errors).
2. `top_k(M, k)` with **deflation**: find the first component, subtract `lam · v v^T` from `M` and repeat. Return eigenvalues (array) and eigenvectors (`k × p` matrix, one per row). Compare with `np.linalg.eigh` (eigenvectors up to the sign).
3. On the wine matrix save `wine_iters` (iterations to converge with `tol = 1e-12`) and on `narrow_C` `narrow_iters`. Save `iteration_ratio = narrow_iters / wine_iters`.
4. **Convergence law:** with `true_v` known, estimate `empirical_rate` as the median of `errors[t+1] / errors[t]` for the iterations with `errors[t] > 1e-10` (the error in `1 − cos` drops like the **square** of the ratio $\lambda_2/\lambda_1$ per iteration: compare `empirical_rate` with `(lam2/lam1) ** 2`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine

Xw = load_wine().data
wine_C = np.corrcoef(Xw.T)
eig, vec = np.linalg.eigh(wine_C)
eig, vec = eig[::-1], vec[:, ::-1]
narrow_eig = eig.copy(); narrow_eig[1] = 0.98 * eig[0]                    # second eigenvalue almost equal to the first
narrow_C = (vec * narrow_eig) @ vec.T
print("lambda2/lambda1 ratio: wine", round(eig[1] / eig[0], 3), "| narrow", round(narrow_eig[1] / narrow_eig[0], 3))

## Your work

In [ ]:
def power(M, tol=1e-12, max_iter=100000, seed=0, true_v=None):
    ...


def top_k(M, k):
    ...

In [ ]:
wine_iters = None
narrow_iters = None
iteration_ratio = None

In [ ]:
empirical_rate = None
theoretical_rate = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`v = rng.normal(size=len(M)); v /= np.linalg.norm(v); while ...: w = M @ v; w /= np.linalg.norm(w); converged if 1 - abs(w @ v) < tol; v = w`. Deflation: `M = M - lam * np.outer(v, v)`. Rate: `e = np.array(errors); e = e[e > 1e-10]; np.median(e[1:] / e[:-1])`.'):
    v, lam, n_it, err = power(wine_C, true_v=vec[:, 0])
    check(abs(v @ vec[:, 0]) > 1 - 1e-10 and close(lam, eig[0]) and len(err) == n_it, "first eigenvector and eigenvalue correct; one error per iteration", "v <- M v / ||M v||")
    lams, vs = top_k(wine_C, 3)
    check(close(lams, eig[:3], rtol=1e-6) and all(abs(vs[i] @ vec[:, i]) > 1 - 1e-8 for i in range(3)), "first three eigenvalues and eigenvectors (up to the sign) match eigh", "deflation: M - lam * outer(v, v)")
    check(wine_iters is not None and wine_iters < 100 and narrow_iters > 20 * wine_iters and close(iteration_ratio, narrow_iters / wine_iters), "wine: fewer than 100 iterations; with the narrow gap over 20 times more", "the convergence rate is (lambda2/lambda1)")
    check(empirical_rate is not None and abs(empirical_rate - theoretical_rate) / theoretical_rate < 0.1 and close(theoretical_rate, (eig[1] / eig[0]) ** 2), "the empirical error rate is (lambda2/lambda1)^2 within 10%", "median of errors[t+1]/errors[t]")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_power_iteration_and_spectral_gap.ipynb`. Open it after you pass the check, or after you have really tried.